In [ ]:
import os

import pandas as pd
import yaml

from pynxtools_apm import SEPARATOR
from pynxtools_apm.examples.deu_berlin_kuehbach.utils import (
    get_project_id,
    prepare_parsing,
)

print(os.getcwd())
with open(f"{os.getcwd()}{os.sep}/config.yaml", encoding="utf-8") as fp:
    config: dict[str, str] = yaml.safe_load(fp)
for task, location in config.items():
    print(f"{task}, {location}")

In [ ]:
spread_sheet_of_all_projects = pd.read_excel(
    f"{config['data']}{os.sep}aaa_legacy_data.ods",
    sheet_name="aaa_legacy_data",
    engine="odf",
    dtype=str,
).fillna("")

project_range: tuple[int, int] = (1, 160)

## Decompress the original files of the scientists and projects from their cold-storage location

Locally, original research data are stored compressed when not needed.<br>
Maybe multiple compressed files per project directory.<br>

In [ ]:
cnt = 0  # how many files to decompress
vol = 0  # how much byte volume does this add to scratch
for row in spread_sheet_of_all_projects.itertuples(index=True):
    if row.project_name != "" and row.legal in ("0", "1") and row.use in ("0", "1"):
        if project_range[0] <= int(row.id) <= project_range[1]:
            project_id = get_project_id(row.id)

            print(
                f"project{SEPARATOR}{row.project_name}{SEPARATOR}{project_id}{SEPARATOR}decompress..."
            )

            status = prepare_parsing(
                row.project_name,
                f"{config['data']}{os.sep}{row.project_name}.ods",
                f"{config['data']}{os.sep}{row.project_name}.sha256.results.csv",
                config["data"],
                config["decompress"],
                report=True,
                write=True,
                logger_file_path_suffix="paper.run01",
            )
            for key, obj in status.items():
                if obj["n"] > 0:
                    # print(f"{project_id}, {key}, {obj['n']}, {obj['bytes']}")
                    cnt += obj["n"]
                    vol += obj["bytes"]

print(f"{cnt} files, if uncompressed {vol / 1024**3} GiB")
print("Batch queue completed")